# Descrição

Este notebook integra os resultados de expressão diferencial (DEG) com os módulos de WGCNA para definir a lista de genes que será enviada ao STRING.

A lógica adotada é:

1. Carregar as tabelas anotadas geradas pelo notebook `004_gene_annotation.ipynb`;
2. Filtrar DEGs com `padj < 0.1` e `|log2FoldChange| >= 1`;
3. Manter apenas contrastes prioritários diretamente relacionados à hipótese do projeto;
4. Integrar DEG com WGCNA por `gene_id → module`;
5. Ranqueiar módulos por:
   - `n_deg_priority`: número de DEGs prioritários no módulo;
   - `fraction_deg_priority`: fração de genes do módulo que são DEGs prioritários;
6. Selecionar os top módulos;
7. Exportar ao STRING apenas os DEGs pertencentes aos módulos selecionados.

A definição dos hubs será feita posteriormente no Cytoscape.

## Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configurações

In [2]:
PROCESSED_DIR = Path("../../data/interim")
ANNOTATION_DIR = PROCESSED_DIR / "annotation"
WGCNA_DIR = PROCESSED_DIR / "wgcna"
STRING_EXPORT_DIR = PROCESSED_DIR / "string_export"

STRING_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Entradas principais produzidas pelo notebook 004
DEG_ALL_ANNOTATED_PATH = ANNOTATION_DIR / "deg_all_contrasts_annotated.csv"
WGCNA_MODULES_ANNOTATED_PATH = ANNOTATION_DIR / "wgcna_gene_modules_annotated.csv"

# Entrada opcional produzida pelo notebook 003:
# usada apenas como anotação auxiliar, não como filtro obrigatório
MODULE_SUMMARY_PATH = WGCNA_DIR / "wgcna_module_summary.csv"

# Saídas
DEG_WGCNA_SELECTED_PATH = STRING_EXPORT_DIR / "deg_wgcna_selected_for_string.csv"
STRING_INPUT_SELECTED_PATH = STRING_EXPORT_DIR / "string_input_selected_genes.txt"
SELECTED_MODULES_SUMMARY_PATH = STRING_EXPORT_DIR / "selected_modules_summary.csv"
CYTOSCAPE_NODE_TABLE_PATH = STRING_EXPORT_DIR / "cytoscape_node_table.csv"

# Critério estatístico
ALPHA = 0.1
ABS_LOG2FC_MIN = 1.0

# Contrastes priorizados para responder à hipótese do projeto
PRIORITY_CONTRASTS = [
    "MA100_vs_CTR",
    "MC1_vs_CTR",
    "MD1_vs_CTR",
    "MA100_vs_MA1",
    "MC100_vs_MC1",
]

# Seleção modular: número máximo de módulos a selecionar (com base no número de DEGs prioritários).
TOP_N_MODULES = 5

# Limiar mínimo de DEGs prioritários para um módulo ser elegível.
MIN_N_DEG_PRIORITY = 5

print("Diretório de exportação:", STRING_EXPORT_DIR)

Diretório de exportação: ../../data/interim/string_export


## Carregamento dos Dados

In [3]:
if not DEG_ALL_ANNOTATED_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DEG_ALL_ANNOTATED_PATH}")

if not WGCNA_MODULES_ANNOTATED_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {WGCNA_MODULES_ANNOTATED_PATH}")

deg_df = pd.read_csv(DEG_ALL_ANNOTATED_PATH)
wgcna_df = pd.read_csv(WGCNA_MODULES_ANNOTATED_PATH)

print("DEG anotado:", deg_df.shape)
print("WGCNA anotado:", wgcna_df.shape)

display(deg_df.head())
display(wgcna_df.head())

DEG anotado: (121740, 12)
WGCNA anotado: (12174, 6)


,gene_id,gene_symbol,gene_name,contrast,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,significant,direction
0,ENSG00000185112,FAM43A,family with sequence similarity 43 member A,MA100_vs_CTR,1161.674476,1.476968,0.188468,7.836697,4.625526e-15,4.102379e-11,True,up_in_MA100
1,ENSG00000135046,ANXA1,annexin A1,MA100_vs_CTR,4815.301901,-0.970120,0.136702,-7.096589,1.278735e-12,5.138287e-09,True,up_in_CTR
2,ENSG00000040275,SPDL1,spindle apparatus coiled-coil protein 1,MA100_vs_CTR,2053.578939,1.214744,0.172206,7.054032,1.738061e-12,5.138287e-09,True,up_in_MA100
3,ENSG00000214357,NEURL1B,neuralized E3 ubiquitin protein ligase 1B,MA100_vs_CTR,1332.742635,1.313576,0.196726,6.677188,2.435707e-11,5.400571e-08,True,up_in_MA100
4,ENSG00000147133,TAF1,TATA-box binding protein associated factor 1,MA100_vs_CTR,1810.470521,1.281281,0.192941,6.640788,3.120115e-11,5.534459e-08,True,up_in_MA100


,gene_id,gene_symbol,gene_name,dynamic_module,module,module_label
0,ENSG00000000003,TSPAN6,tetraspanin 6,dimgrey,dimgrey,11
1,ENSG00000000419,DPM1,dolichyl-phosphate mannosyltransferase subunit...,silver,silver,29
2,ENSG00000000457,SCYL3,SCY1 like pseudokinase 3,dimgrey,dimgrey,11
3,ENSG00000000460,FIRRM,FIGNL1 interacting regulator of recombination ...,dimgrey,dimgrey,11
4,ENSG00000001036,FUCA2,alpha-L-fucosidase 2,lightcoral,lightcoral,14


In [4]:
# Garante uma única linha de módulo por gene
wgcna_df = (
    wgcna_df
    .sort_values(["gene_id"])
    .drop_duplicates(subset="gene_id", keep="first")
    .reset_index(drop=True)
)

print("Genes únicos em DEG:", deg_df["gene_id"].nunique())
print("Genes únicos em WGCNA:", wgcna_df["gene_id"].nunique())
print("Contrastes em DEG:", deg_df["contrast"].nunique())

Genes únicos em DEG: 12174
Genes únicos em WGCNA: 12174
Contrastes em DEG: 10


## Integração DEG x WGCNA

In [5]:
# Evita colisão de colunas de anotação.
# Preferimos as anotações da tabela DEG quando já existem.

wgcna_cols_to_add = [
    col for col in wgcna_df.columns
    if col not in {"gene_symbol", "gene_name"}
]

integrated_df = deg_df.merge(
    wgcna_df[wgcna_cols_to_add],
    on="gene_id",
    how="left",
    validate="many_to_one",
)

integrated_df["module"] = integrated_df["module"].astype("string")

integrated_df["log2FoldChange"] = pd.to_numeric(
    integrated_df["log2FoldChange"],
    errors="coerce"
)

integrated_df["pvalue"] = pd.to_numeric(
    integrated_df["pvalue"],
    errors="coerce"
)

integrated_df["padj"] = pd.to_numeric(
    integrated_df["padj"],
    errors="coerce"
)

integrated_df["abs_log2FoldChange"] = integrated_df["log2FoldChange"].abs()

padj_for_log = integrated_df["padj"].replace(0, np.nextafter(0, 1))
integrated_df["neg_log10_padj"] = -np.log10(padj_for_log)

# Fallback para symbol/label quando a anotação estiver ausente
if "gene_symbol" not in integrated_df.columns:
    integrated_df["gene_symbol"] = np.nan

integrated_df["is_deg"] = (
    (integrated_df["padj"] < ALPHA)
    & integrated_df["padj"].notna()
    & (integrated_df["abs_log2FoldChange"] >= ABS_LOG2FC_MIN)
)

integrated_df["is_priority_contrast"] = integrated_df["contrast"].isin(PRIORITY_CONTRASTS)

# Sem exclusão automática de módulos grey/dimgrey/etc.
# O requisito é apenas o gene possuir módulo WGCNA informado.
integrated_df["has_wgcna_module"] = integrated_df["module"].notna()

integrated_df["regulation"] = np.select(
    [
        integrated_df["log2FoldChange"] > 0,
        integrated_df["log2FoldChange"] < 0,
    ],
    [
        "up_in_tested",
        "up_in_reference",
    ],
    default="unchanged_or_zero",
)

integrated_df["selected_priority_deg"] = (
    integrated_df["is_deg"]
    & integrated_df["is_priority_contrast"]
    & integrated_df["has_wgcna_module"]
)

print("Tabela integrada:", integrated_df.shape)
print("Genes únicos integrados:", integrated_df["gene_id"].nunique())
print("Linhas DEG significativas:", int(integrated_df["is_deg"].sum()))
print("Linhas DEG prioritárias com módulo WGCNA:", int(integrated_df["selected_priority_deg"].sum()))
print("Genes únicos DEG prioritários com módulo WGCNA:", integrated_df.loc[integrated_df["selected_priority_deg"], "gene_id"].nunique())

display(integrated_df.head())

Tabela integrada: (121740, 22)
Genes únicos integrados: 12174
Linhas DEG significativas: 873
Linhas DEG prioritárias com módulo WGCNA: 846
Genes únicos DEG prioritários com módulo WGCNA: 607


,gene_id,gene_symbol,gene_name,contrast,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,...,dynamic_module,module,module_label,abs_log2FoldChange,neg_log10_padj,is_deg,is_priority_contrast,has_wgcna_module,regulation,selected_priority_deg
0,ENSG00000185112,FAM43A,family with sequence similarity 43 member A,MA100_vs_CTR,1161.674476,1.476968,0.188468,7.836697,4.625526e-15,4.102379e-11,...,darkgrey,darkgrey,7,1.476968,10.386964,True,True,True,up_in_tested,True
1,ENSG00000135046,ANXA1,annexin A1,MA100_vs_CTR,4815.301901,-0.970120,0.136702,-7.096589,1.278735e-12,5.138287e-09,...,silver,silver,29,0.970120,8.289182,False,True,True,up_in_reference,False
2,ENSG00000040275,SPDL1,spindle apparatus coiled-coil protein 1,MA100_vs_CTR,2053.578939,1.214744,0.172206,7.054032,1.738061e-12,5.138287e-09,...,dimgrey,dimgrey,11,1.214744,8.289182,True,True,True,up_in_tested,True
3,ENSG00000214357,NEURL1B,neuralized E3 ubiquitin protein ligase 1B,MA100_vs_CTR,1332.742635,1.313576,0.196726,6.677188,2.435707e-11,5.400571e-08,...,dimgrey,dimgrey,11,1.313576,7.267560,True,True,True,up_in_tested,True
4,ENSG00000147133,TAF1,TATA-box binding protein associated factor 1,MA100_vs_CTR,1810.470521,1.281281,0.192941,6.640788,3.120115e-11,5.534459e-08,...,darkgrey,darkgrey,7,1.281281,7.256925,True,True,True,up_in_tested,True


## Tabela de DEGs prioritários

In [6]:
priority_deg_df = integrated_df.loc[integrated_df["selected_priority_deg"]].copy()

if priority_deg_df.empty:
    raise ValueError(
        "Nenhum DEG prioritário foi encontrado com os critérios atuais. "
        "Verifique ALPHA, ABS_LOG2FC_MIN e PRIORITY_CONTRASTS."
    )

print("DEGs prioritários:", priority_deg_df.shape)
print("Genes únicos:", priority_deg_df["gene_id"].nunique())

display(priority_deg_df.head())

DEGs prioritários: (846, 22)
Genes únicos: 607


,gene_id,gene_symbol,gene_name,contrast,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,...,dynamic_module,module,module_label,abs_log2FoldChange,neg_log10_padj,is_deg,is_priority_contrast,has_wgcna_module,regulation,selected_priority_deg
0,ENSG00000185112,FAM43A,family with sequence similarity 43 member A,MA100_vs_CTR,1161.674476,1.476968,0.188468,7.836697,4.625526e-15,4.102379e-11,...,darkgrey,darkgrey,7,1.476968,10.386964,True,True,True,up_in_tested,True
2,ENSG00000040275,SPDL1,spindle apparatus coiled-coil protein 1,MA100_vs_CTR,2053.578939,1.214744,0.172206,7.054032,1.738061e-12,5.138287e-09,...,dimgrey,dimgrey,11,1.214744,8.289182,True,True,True,up_in_tested,True
3,ENSG00000214357,NEURL1B,neuralized E3 ubiquitin protein ligase 1B,MA100_vs_CTR,1332.742635,1.313576,0.196726,6.677188,2.435707e-11,5.400571e-08,...,dimgrey,dimgrey,11,1.313576,7.267560,True,True,True,up_in_tested,True
4,ENSG00000147133,TAF1,TATA-box binding protein associated factor 1,MA100_vs_CTR,1810.470521,1.281281,0.192941,6.640788,3.120115e-11,5.534459e-08,...,darkgrey,darkgrey,7,1.281281,7.256925,True,True,True,up_in_tested,True
5,ENSG00000131747,TOP2A,DNA topoisomerase II alpha,MA100_vs_CTR,19915.706763,1.038259,0.157207,6.604396,3.991402e-11,5.899957e-08,...,dimgrey,dimgrey,11,1.038259,7.229151,True,True,True,up_in_tested,True


## Ranqueamento dos Módulos WGCNA

In [7]:
module_rows = []

for module_name, module_df in integrated_df.groupby("module", dropna=False):
    if pd.isna(module_name):
        continue

    module_priority_df = priority_deg_df.loc[priority_deg_df["module"] == module_name]

    priority_contrasts_present = (
        module_priority_df["contrast"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    n_unique_genes_in_module = module_df["gene_id"].nunique()
    n_deg_priority = module_priority_df["gene_id"].nunique()

    module_rows.append({
        "module": module_name,
        "n_unique_genes_in_module": n_unique_genes_in_module,
        "n_deg_priority": n_deg_priority,
        "fraction_deg_priority": (
            n_deg_priority / n_unique_genes_in_module
            if n_unique_genes_in_module > 0
            else np.nan
        ),
        "n_priority_gene_contrast_hits": module_priority_df.shape[0],
        "n_priority_contrasts_present": len(priority_contrasts_present),
        "priority_contrasts_present": ";".join(priority_contrasts_present),
        "n_up_priority_rows": int((module_priority_df["regulation"] == "up_in_tested").sum()),
        "n_down_priority_rows": int((module_priority_df["regulation"] == "up_in_reference").sum()),
    })

module_ranking_df = (
    pd.DataFrame(module_rows)
    .sort_values(
        [
            "n_deg_priority",
            "fraction_deg_priority",
            "n_priority_contrasts_present",
        ],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

display(module_ranking_df.head(20))

,module,n_unique_genes_in_module,n_deg_priority,fraction_deg_priority,n_priority_gene_contrast_hits,n_priority_contrasts_present,priority_contrasts_present,n_up_priority_rows,n_down_priority_rows
0,dimgrey,3847,299,0.077723,437,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,201,236
1,darkgrey,3279,202,0.061604,262,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,114,148
2,gainsboro,1116,43,0.038530,72,4,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,47,25
3,silver,2145,39,0.018182,45,4,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_CTR,35,10
4,lightcoral,192,6,0.031250,6,1,MC100_vs_MC1,6,0
5,white,316,6,0.018987,7,4,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,3,4
6,firebrick,171,3,0.017544,4,4,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MD1_vs_CTR,3,1
7,sienna,27,2,0.074074,2,2,MC100_vs_MC1;MD1_vs_CTR,2,0
8,maroon,340,2,0.005882,6,4,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,5,1
9,moccasin,8,1,0.125000,1,1,MA100_vs_CTR,1,0


## Seleção de Módulos

In [8]:
eligible_modules_df = module_ranking_df.loc[
    module_ranking_df["n_deg_priority"] >= MIN_N_DEG_PRIORITY
].copy()

if eligible_modules_df.empty:
    raise ValueError(
        "Nenhum módulo atingiu MIN_N_DEG_PRIORITY. "
        "Reduza MIN_N_DEG_PRIORITY ou revise os critérios DEG."
    )

selected_modules_df = (
    eligible_modules_df
    .head(TOP_N_MODULES)
    .copy()
)

selected_modules = selected_modules_df["module"].astype(str).tolist()

print(f"Top {TOP_N_MODULES} módulos selecionados, com mínimo de {MIN_N_DEG_PRIORITY} DEG(s) prioritário(s):")
print(selected_modules)

display(selected_modules_df)

Top 5 módulos selecionados, com mínimo de 5 DEG(s) prioritário(s):
['dimgrey', 'darkgrey', 'gainsboro', 'silver', 'lightcoral']


,module,n_unique_genes_in_module,n_deg_priority,fraction_deg_priority,n_priority_gene_contrast_hits,n_priority_contrasts_present,priority_contrasts_present,n_up_priority_rows,n_down_priority_rows
0,dimgrey,3847,299,0.077723,437,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,201,236
1,darkgrey,3279,202,0.061604,262,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,114,148
2,gainsboro,1116,43,0.038530,72,4,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,47,25
3,silver,2145,39,0.018182,45,4,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_CTR,35,10
4,lightcoral,192,6,0.031250,6,1,MC100_vs_MC1,6,0


## Seleção Final de Genes

In [9]:
selected_df = priority_deg_df.loc[
    priority_deg_df["module"].astype(str).isin(selected_modules)
].copy()

if selected_df.empty:
    raise ValueError(
        "Nenhum gene foi selecionado para STRING após o filtro por módulos selecionados."
    )

# Nome usado para STRING e para mapeamento inicial no Cytoscape.
# Preferimos gene_symbol, mas mantemos gene_id como fallback.
selected_df["string_gene_label"] = selected_df["gene_symbol"].fillna(selected_df["gene_id"])

group_cols = ["gene_id", "gene_symbol", "string_gene_label"]

agg_dict = {
    "modules": (
        "module",
        lambda x: ";".join(
            x.dropna().astype(str).drop_duplicates().sort_values().tolist()
        ),
    ),
    "contrasts": (
        "contrast",
        lambda x: ";".join(
            x.dropna().astype(str).drop_duplicates().sort_values().tolist()
        ),
    ),
    "n_selected_rows": ("gene_id", "size"),
    "n_contrasts": ("contrast", "nunique"),
    "min_padj": ("padj", "min"),
    "max_abs_log2FoldChange": ("abs_log2FoldChange", "max"),
    "mean_abs_log2FoldChange": ("abs_log2FoldChange", "mean"),
    "n_up_rows": (
        "regulation",
        lambda x: int((x == "up_in_tested").sum()),
    ),
    "n_down_rows": (
        "regulation",
        lambda x: int((x == "up_in_reference").sum()),
    ),
}

if "gene_name" in selected_df.columns:
    agg_dict["gene_name"] = ("gene_name", "first")

string_gene_table_df = (
    selected_df
    .groupby(group_cols, dropna=False)
    .agg(**agg_dict)
    .reset_index()
    .sort_values(
        ["min_padj", "max_abs_log2FoldChange"],
        ascending=[True, False],
    )
)

# Lista final: uma entrada por linha para colar/importar no STRING.
string_gene_list = (
    string_gene_table_df["string_gene_label"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

# Linha de maior efeito absoluto por gene:
# útil para colorir ou priorizar nós no Cytoscape.
strongest_effect_df = (
    selected_df
    .sort_values(
        ["gene_id", "abs_log2FoldChange", "padj"],
        ascending=[True, False, True],
    )
    .drop_duplicates(subset="gene_id", keep="first")
    [[
        "gene_id",
        "contrast",
        "log2FoldChange",
        "padj",
        "regulation",
    ]]
    .rename(columns={
        "contrast": "strongest_contrast",
        "log2FoldChange": "strongest_log2FoldChange",
        "padj": "strongest_padj",
        "regulation": "strongest_regulation",
    })
)

# Matriz larga por contraste:
# facilita mapear cor/tamanho no Cytoscape por contraste específico.
log2fc_wide_df = (
    selected_df
    .pivot_table(
        index="gene_id",
        columns="contrast",
        values="log2FoldChange",
        aggfunc="first",
    )
)

log2fc_wide_df.columns = [
    f"log2FC__{col}" for col in log2fc_wide_df.columns
]

log2fc_wide_df = log2fc_wide_df.reset_index()

padj_wide_df = (
    selected_df
    .pivot_table(
        index="gene_id",
        columns="contrast",
        values="padj",
        aggfunc="first",
    )
)

padj_wide_df.columns = [
    f"padj__{col}" for col in padj_wide_df.columns
]

padj_wide_df = padj_wide_df.reset_index()

regulation_wide_df = (
    selected_df
    .pivot_table(
        index="gene_id",
        columns="contrast",
        values="regulation",
        aggfunc=lambda x: ";".join(
            x.dropna().astype(str).drop_duplicates().sort_values().tolist()
        ),
    )
)

regulation_wide_df.columns = [
    f"regulation__{col}" for col in regulation_wide_df.columns
]

regulation_wide_df = regulation_wide_df.reset_index()

# Tabela final de atributos para nós no Cytoscape.
# Deve ter uma linha por gene/nó.
cytoscape_node_table_df = (
    string_gene_table_df
    .merge(strongest_effect_df, on="gene_id", how="left", validate="one_to_one")
    .merge(log2fc_wide_df, on="gene_id", how="left", validate="one_to_one")
    .merge(padj_wide_df, on="gene_id", how="left", validate="one_to_one")
    .merge(regulation_wide_df, on="gene_id", how="left", validate="one_to_one")
)

# Coluna principal para mapear com os nós importados do STRING.
# No Cytoscape, normalmente deve ser usada para matching contra o nome/display name do nó.
cytoscape_node_table_df.insert(
    0,
    "node_name",
    cytoscape_node_table_df["string_gene_label"]
)

# Organização básica das colunas principais.
main_cols = [
    "node_name",
    "gene_id",
    "gene_symbol",
    "string_gene_label",
]

if "gene_name" in cytoscape_node_table_df.columns:
    main_cols.append("gene_name")

main_cols += [
    "modules",
    "contrasts",
    "n_selected_rows",
    "n_contrasts",
    "min_padj",
    "max_abs_log2FoldChange",
    "mean_abs_log2FoldChange",
    "strongest_contrast",
    "strongest_log2FoldChange",
    "strongest_padj",
    "strongest_regulation",
    "n_up_rows",
    "n_down_rows",
]

remaining_cols = [
    col for col in cytoscape_node_table_df.columns
    if col not in main_cols
]

cytoscape_node_table_df = cytoscape_node_table_df[
    main_cols + remaining_cols
]

print("Critério final:")
print(f"- padj < {ALPHA}")
print(f"- |log2FoldChange| >= {ABS_LOG2FC_MIN}")
print(f"- contrastes prioritários: {PRIORITY_CONTRASTS}")
print(f"- módulos selecionados: {selected_modules}")

print("\nLinhas selecionadas:", selected_df.shape[0])
print("Genes únicos selecionados:", selected_df["gene_id"].nunique())
print("Entradas únicas para STRING:", len(string_gene_list))
print("Linhas na tabela de nós do Cytoscape:", cytoscape_node_table_df.shape[0])

print("\nPrimeiros genes para STRING:")
print("\n".join(string_gene_list[:30]))

display(string_gene_table_df.head(30))
display(cytoscape_node_table_df.head(30))

Critério final:
- padj < 0.1
- |log2FoldChange| >= 1.0
- contrastes prioritários: ['MA100_vs_CTR', 'MC1_vs_CTR', 'MD1_vs_CTR', 'MA100_vs_MA1', 'MC100_vs_MC1']
- módulos selecionados: ['dimgrey', 'darkgrey', 'gainsboro', 'silver', 'lightcoral']

Linhas selecionadas: 822
Genes únicos selecionados: 589
Entradas únicas para STRING: 589
Linhas na tabela de nós do Cytoscape: 589

Primeiros genes para STRING:
FAM43A
KHSRP
SPDL1
TOP2A
CAVIN1
TAF1
GANAB
PTPN1
NFIX
UBE2Z
RPSA
SPON2
EEF1A1
MAP2K3
RPL12
TMEM35A
FXR2
RRBP1
CETN3
NDST1
HOMER3
CBX3
TTYH3
R3HDM4
RPS15A
CKAP2L
TMEM134
NEURL1B
HDAC7
SPCS1


,gene_id,gene_symbol,string_gene_label,modules,contrasts,n_selected_rows,n_contrasts,min_padj,max_abs_log2FoldChange,mean_abs_log2FoldChange,n_up_rows,n_down_rows,gene_name
475,ENSG00000185112,FAM43A,FAM43A,darkgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,4,4,1.153330e-12,1.557119,1.326396,3,1,family with sequence similarity 43 member A
55,ENSG00000088247,KHSRP,KHSRP,dimgrey,MC100_vs_MC1,1,1,2.877030e-11,1.285579,1.285579,0,1,KH-type splicing regulatory protein
22,ENSG00000040275,SPDL1,SPDL1,dimgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,3,3,3.272197e-11,1.333603,1.232687,2,1,spindle apparatus coiled-coil protein 1
185,ENSG00000131747,TOP2A,TOP2A,dimgrey,MA100_vs_CTR;MC100_vs_MC1,2,2,8.612006e-11,1.186218,1.112239,1,1,DNA topoisomerase II alpha
434,ENSG00000177469,CAVIN1,CAVIN1,darkgrey,MC100_vs_MC1,1,1,8.612006e-11,1.060500,1.060500,0,1,caveolae associated protein 1
262,ENSG00000147133,TAF1,TAF1,darkgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,4,4,1.335214e-10,1.434945,1.329090,3,1,TATA-box binding protein associated factor 1
57,ENSG00000089597,GANAB,GANAB,darkgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,3,3,1.335214e-10,1.366875,1.203165,2,1,glucosidase II alpha subunit
497,ENSG00000196396,PTPN1,PTPN1,darkgrey,MC100_vs_MC1;MC1_vs_CTR,2,2,1.543837e-10,1.165922,1.089112,1,1,protein tyrosine phosphatase non-receptor type 1
6,ENSG00000008441,NFIX,NFIX,darkgrey,MC100_vs_MC1,1,1,8.778300e-10,1.247167,1.247167,0,1,nuclear factor I X
308,ENSG00000159202,UBE2Z,UBE2Z,dimgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,3,3,1.360688e-09,1.404989,1.315906,2,1,ubiquitin conjugating enzyme E2 Z


,node_name,gene_id,gene_symbol,string_gene_label,gene_name,modules,contrasts,n_selected_rows,n_contrasts,min_padj,...,padj__MA100_vs_CTR,padj__MA100_vs_MA1,padj__MC100_vs_MC1,padj__MC1_vs_CTR,padj__MD1_vs_CTR,regulation__MA100_vs_CTR,regulation__MA100_vs_MA1,regulation__MC100_vs_MC1,regulation__MC1_vs_CTR,regulation__MD1_vs_CTR
0,FAM43A,ENSG00000185112,FAM43A,FAM43A,family with sequence similarity 43 member A,darkgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,4,4,1.153330e-12,...,4.102379e-11,NaN,4.950792e-07,1.153330e-12,4.622344e-07,up_in_tested,NaN,up_in_reference,up_in_tested,up_in_tested
1,KHSRP,ENSG00000088247,KHSRP,KHSRP,KH-type splicing regulatory protein,dimgrey,MC100_vs_MC1,1,1,2.877030e-11,...,NaN,NaN,2.877030e-11,NaN,NaN,NaN,NaN,up_in_reference,NaN,NaN
2,SPDL1,ENSG00000040275,SPDL1,SPDL1,spindle apparatus coiled-coil protein 1,dimgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,3,3,3.272197e-11,...,5.138287e-09,NaN,3.272197e-11,3.478633e-08,NaN,up_in_tested,NaN,up_in_reference,up_in_tested,NaN
3,TOP2A,ENSG00000131747,TOP2A,TOP2A,DNA topoisomerase II alpha,dimgrey,MA100_vs_CTR;MC100_vs_MC1,2,2,8.612006e-11,...,5.899957e-08,NaN,8.612006e-11,NaN,NaN,up_in_tested,NaN,up_in_reference,NaN,NaN
4,CAVIN1,ENSG00000177469,CAVIN1,CAVIN1,caveolae associated protein 1,darkgrey,MC100_vs_MC1,1,1,8.612006e-11,...,NaN,NaN,8.612006e-11,NaN,NaN,NaN,NaN,up_in_reference,NaN,NaN
5,TAF1,ENSG00000147133,TAF1,TAF1,TATA-box binding protein associated factor 1,darkgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,4,4,1.335214e-10,...,5.534459e-08,NaN,1.335214e-10,2.009255e-07,2.600575e-09,up_in_tested,NaN,up_in_reference,up_in_tested,up_in_tested
6,GANAB,ENSG00000089597,GANAB,GANAB,glucosidase II alpha subunit,darkgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,3,3,1.335214e-10,...,4.772668e-07,NaN,1.335214e-10,7.875974e-07,NaN,up_in_tested,NaN,up_in_reference,up_in_tested,NaN
7,PTPN1,ENSG00000196396,PTPN1,PTPN1,protein tyrosine phosphatase non-receptor type 1,darkgrey,MC100_vs_MC1;MC1_vs_CTR,2,2,1.543837e-10,...,NaN,NaN,1.543837e-10,1.610333e-07,NaN,NaN,NaN,up_in_reference,up_in_tested,NaN
8,NFIX,ENSG00000008441,NFIX,NFIX,nuclear factor I X,darkgrey,MC100_vs_MC1,1,1,8.778300e-10,...,NaN,NaN,8.778300e-10,NaN,NaN,NaN,NaN,up_in_reference,NaN,NaN
9,UBE2Z,ENSG00000159202,UBE2Z,UBE2Z,ubiquitin conjugating enzyme E2 Z,dimgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,3,3,1.360688e-09,...,8.650208e-08,NaN,1.686202e-08,1.360688e-09,NaN,up_in_tested,NaN,up_in_reference,up_in_tested,NaN


In [10]:
# Resumo por contraste

contrast_rows = []

for contrast, contrast_df in selected_df.groupby("contrast", dropna=False):
    contrast_rows.append({
        "contrast": contrast,
        "n_selected_rows": contrast_df.shape[0],
        "n_selected_unique_genes": contrast_df["gene_id"].nunique(),
        "n_modules_present": contrast_df["module"].nunique(),
        "modules_present": ";".join(
            contrast_df["module"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .sort_values()
            .tolist()
        ),
        "n_up_rows": int((contrast_df["regulation"] == "up_in_tested").sum()),
        "n_down_rows": int((contrast_df["regulation"] == "up_in_reference").sum()),
    })

summary_by_contrast_df = (
    pd.DataFrame(contrast_rows)
    .sort_values("n_selected_unique_genes", ascending=False)
    .reset_index(drop=True)
)

display(summary_by_contrast_df)

,contrast,n_selected_rows,n_selected_unique_genes,n_modules_present,modules_present,n_up_rows,n_down_rows
0,MC100_vs_MC1,409,409,5,darkgrey;dimgrey;gainsboro;lightcoral;silver,195,214
1,MA100_vs_CTR,167,167,4,darkgrey;dimgrey;gainsboro;silver,78,89
2,MC1_vs_CTR,147,147,4,darkgrey;dimgrey;gainsboro;silver,73,74
3,MD1_vs_CTR,91,91,3,darkgrey;dimgrey;gainsboro,49,42
4,MA100_vs_MA1,8,8,3,darkgrey;dimgrey;silver,8,0


In [11]:
# Resumo por módulo

selected_module_rows = []

for module_name, module_df in selected_df.groupby("module", dropna=False):
    contrasts_present = (
        module_df["contrast"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    selected_module_rows.append({
        "module": module_name,
        "n_selected_rows": module_df.shape[0],
        "n_selected_unique_genes": module_df["gene_id"].nunique(),
        "n_contrasts_present": len(contrasts_present),
        "contrasts_present": ";".join(contrasts_present),
        "n_up_rows": int((module_df["regulation"] == "up_in_tested").sum()),
        "n_down_rows": int((module_df["regulation"] == "up_in_reference").sum()),
    })

summary_by_module_df = (
    pd.DataFrame(selected_module_rows)
    .sort_values("n_selected_unique_genes", ascending=False)
    .reset_index(drop=True)
)

display(summary_by_module_df)

,module,n_selected_rows,n_selected_unique_genes,n_contrasts_present,contrasts_present,n_up_rows,n_down_rows
0,dimgrey,437,299,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,201,236
1,darkgrey,262,202,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,114,148
2,gainsboro,72,43,4,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,47,25
3,silver,45,39,4,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_CTR,35,10
4,lightcoral,6,6,1,MC100_vs_MC1,6,0


## Salvamento

In [12]:
selected_df.to_csv(DEG_WGCNA_SELECTED_PATH, index=False)

selected_modules_df.to_csv(SELECTED_MODULES_SUMMARY_PATH, index=False)

cytoscape_node_table_df.to_csv(CYTOSCAPE_NODE_TABLE_PATH, index=False)

with open(STRING_INPUT_SELECTED_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(string_gene_list))
    f.write("\n")

print("Arquivos finais salvos:")
print("-", DEG_WGCNA_SELECTED_PATH)
print("-", SELECTED_MODULES_SUMMARY_PATH)
print("-", CYTOSCAPE_NODE_TABLE_PATH)
print("-", STRING_INPUT_SELECTED_PATH)

Arquivos finais salvos:
- ../../data/interim/string_export/deg_wgcna_selected_for_string.csv
- ../../data/interim/string_export/selected_modules_summary.csv
- ../../data/interim/string_export/cytoscape_node_table.csv
- ../../data/interim/string_export/string_input_selected_genes.txt


## Resumo

In [13]:
print("Resumo final")
print("============")
print(f"Genes únicos DEG prioritários antes do filtro modular: {priority_deg_df['gene_id'].nunique()}")
print(f"Top módulos selecionados: {selected_modules}")
print(f"Genes únicos selecionados para STRING: {len(string_gene_list)}")
print(f"Contrastes representados: {selected_df['contrast'].nunique()}")
print(f"Módulos representados: {selected_df['module'].nunique()}")

print("\nMódulos selecionados:")
display(selected_modules_df)

print("\nGenes selecionados por módulo:")
display(
    selected_df
    .groupby("module")
    .agg(n_genes=("gene_id", "nunique"))
    .reset_index()
    .sort_values("n_genes", ascending=False)
)

print("\nGenes selecionados por contraste:")
display(
    selected_df
    .groupby("contrast")
    .agg(n_genes=("gene_id", "nunique"))
    .reset_index()
    .sort_values("n_genes", ascending=False)
)

print("\nArquivo principal para STRING:")
print(STRING_INPUT_SELECTED_PATH)

print("\nArquivo de atributos dos nós para Cytoscape:")
print(CYTOSCAPE_NODE_TABLE_PATH)

print("\nResumo da tabela de nós do Cytoscape:")
print("Linhas:", cytoscape_node_table_df.shape[0])
print("Colunas:", cytoscape_node_table_df.shape[1])
print("Nós únicos:", cytoscape_node_table_df["node_name"].nunique())

display(cytoscape_node_table_df.head())

Resumo final
Genes únicos DEG prioritários antes do filtro modular: 607
Top módulos selecionados: ['dimgrey', 'darkgrey', 'gainsboro', 'silver', 'lightcoral']
Genes únicos selecionados para STRING: 589
Contrastes representados: 5
Módulos representados: 5

Módulos selecionados:


,module,n_unique_genes_in_module,n_deg_priority,fraction_deg_priority,n_priority_gene_contrast_hits,n_priority_contrasts_present,priority_contrasts_present,n_up_priority_rows,n_down_priority_rows
0,dimgrey,3847,299,0.077723,437,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,201,236
1,darkgrey,3279,202,0.061604,262,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,114,148
2,gainsboro,1116,43,0.038530,72,4,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,47,25
3,silver,2145,39,0.018182,45,4,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_CTR,35,10
4,lightcoral,192,6,0.031250,6,1,MC100_vs_MC1,6,0



Genes selecionados por módulo:


,module,n_genes
1,dimgrey,299
0,darkgrey,202
2,gainsboro,43
4,silver,39
3,lightcoral,6



Genes selecionados por contraste:


,contrast,n_genes
2,MC100_vs_MC1,409
0,MA100_vs_CTR,167
3,MC1_vs_CTR,147
4,MD1_vs_CTR,91
1,MA100_vs_MA1,8



Arquivo principal para STRING:
../../data/interim/string_export/string_input_selected_genes.txt

Arquivo de atributos dos nós para Cytoscape:
../../data/interim/string_export/cytoscape_node_table.csv

Resumo da tabela de nós do Cytoscape:
Linhas: 589
Colunas: 33
Nós únicos: 589


,node_name,gene_id,gene_symbol,string_gene_label,gene_name,modules,contrasts,n_selected_rows,n_contrasts,min_padj,...,padj__MA100_vs_CTR,padj__MA100_vs_MA1,padj__MC100_vs_MC1,padj__MC1_vs_CTR,padj__MD1_vs_CTR,regulation__MA100_vs_CTR,regulation__MA100_vs_MA1,regulation__MC100_vs_MC1,regulation__MC1_vs_CTR,regulation__MD1_vs_CTR
0,FAM43A,ENSG00000185112,FAM43A,FAM43A,family with sequence similarity 43 member A,darkgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,4,4,1.153330e-12,...,4.102379e-11,NaN,4.950792e-07,1.153330e-12,4.622344e-07,up_in_tested,NaN,up_in_reference,up_in_tested,up_in_tested
1,KHSRP,ENSG00000088247,KHSRP,KHSRP,KH-type splicing regulatory protein,dimgrey,MC100_vs_MC1,1,1,2.877030e-11,...,NaN,NaN,2.877030e-11,NaN,NaN,NaN,NaN,up_in_reference,NaN,NaN
2,SPDL1,ENSG00000040275,SPDL1,SPDL1,spindle apparatus coiled-coil protein 1,dimgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,3,3,3.272197e-11,...,5.138287e-09,NaN,3.272197e-11,3.478633e-08,NaN,up_in_tested,NaN,up_in_reference,up_in_tested,NaN
3,TOP2A,ENSG00000131747,TOP2A,TOP2A,DNA topoisomerase II alpha,dimgrey,MA100_vs_CTR;MC100_vs_MC1,2,2,8.612006e-11,...,5.899957e-08,NaN,8.612006e-11,NaN,NaN,up_in_tested,NaN,up_in_reference,NaN,NaN
4,CAVIN1,ENSG00000177469,CAVIN1,CAVIN1,caveolae associated protein 1,darkgrey,MC100_vs_MC1,1,1,8.612006e-11,...,NaN,NaN,8.612006e-11,NaN,NaN,NaN,NaN,up_in_reference,NaN,NaN
